# CineDetetive — embeddings dos trechos na GPU

**Para a busca por sentido funcionar, cada trecho de sinopse precisa virar um vetor de 1024 números.** Na CPU local seriam ~32 h; numa GPU T4, uns 20–30 min.

Este notebook faz uma coisa só: **recebe texto, devolve vetor.** A limpeza e a quebra em trechos ficam no repositório (`ingestion/chunk_load.py`).

**Antes de rodar:**
1. *Ambiente de execução → Alterar o tipo de ambiente de execução → **T4 GPU***
2. No seu Google Drive, crie a pasta **`cinedetetive`** e suba nela o arquivo **`data/colab/chunks.parquet`**

**Saída:** `cinedetetive/embeddings_bge-m3.npz` no Drive (baixe para `data/colab/` no seu computador).

## 1. Conferir a GPU
Deve aparecer **Tesla T4**. Se der erro, a GPU não foi ativada (veja o item 1 acima).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Ler os trechos do Google Drive
O Colab vai pedir permissão para acessar o seu Drive.

In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')
PASTA = '/content/drive/MyDrive/cinedetetive'

trechos = pd.read_parquet(f'{PASTA}/chunks.parquet')
print(f'{len(trechos):,} trechos')
trechos.head(3)

## 3. Carregar o modelo
**O mesmo modelo que o Ollama usa nas perguntas** (`BAAI/bge-m3`). Se fosse outro, os vetores ficariam em "mapas" diferentes e a busca não acharia nada — sem dar erro.

- `half()`: meia precisão (float16) — 2x mais rápido na GPU, diferença desprezível no resultado
- `max_seq_length = 512`: os trechos têm no máximo ~950 caracteres (~250 tokens), sobra folga

In [ ]:
from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer('BAAI/bge-m3', device='cuda')
modelo.half()
modelo.max_seq_length = 512
print(modelo)
print('dimensões:', modelo.get_sentence_embedding_dimension())

## 4. Gerar os embeddings
- `normalize_embeddings=True`: todo vetor fica com tamanho 1 (o Ollama também normaliza)
- Ordenar por tamanho antes agrupa textos parecidos no mesmo lote → menos desperdício na GPU

⏱️ ~20–30 min. **Não feche a aba** — o Colab gratuito desconecta se ficar muito tempo sem interação.

In [ ]:
import time
import numpy as np

ordem = trechos['content'].str.len().sort_values().index
textos = trechos.loc[ordem, 'content'].tolist()
ids = trechos.loc[ordem, 'id'].to_numpy()

inicio = time.time()
vetores = modelo.encode(
    textos,
    batch_size=128,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
)
minutos = (time.time() - inicio) / 60
print(f'{len(vetores):,} vetores de {vetores.shape[1]} dimensões em {minutos:.1f} min ({len(vetores) / (minutos * 60):.0f} trechos/s)')

## 5. Conferir antes de salvar

In [ ]:
normas = np.linalg.norm(vetores.astype('float32'), axis=1)
print('formato:', vetores.shape)
print('algum NaN?', bool(np.isnan(vetores).any()))
print(f'tamanho dos vetores: min {normas.min():.3f}, max {normas.max():.3f}  (devem ser ~1.0)')

# Sanidade: o trecho mais parecido com uma pergunta tem que fazer sentido
pergunta = modelo.encode(['a man relives the same day over and over'], normalize_embeddings=True)[0]
melhores = np.argsort(-(vetores @ pergunta))[:3]
for i in melhores:
    print('\n→', textos[i][:160].replace('\n', ' | '))

## 6. Salvar no Drive
float16 ocupa metade do espaço (~240 MB) e é suficiente para similaridade de cosseno.

Depois: no Google Drive, baixe `cinedetetive/embeddings_bge-m3.npz` e coloque em **`data/colab/`** no seu computador.

In [ ]:
saida = f'{PASTA}/embeddings_bge-m3.npz'
np.savez(saida, ids=ids.astype('int64'), vetores=vetores.astype('float16'), modelo=np.array('BAAI/bge-m3'))

import os
print(f'Salvo: {saida} ({os.path.getsize(saida) / 1e6:.0f} MB)')